[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No-Country-simulation/G9-LATAM-Team71/blob/develop/data-science/Finance_AI_Asistente_Inteligente_de_Salud_Financiera.ipynb)

# Notebook Finance AI - Asistente Inteligente de Salud Financiera

## Librerias necesarias

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import requests
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold,cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline as imbpipeline
from imblearn.over_sampling import SMOTE

Obtención de datos 

In [ ]:
url = 'https://raw.githubusercontent.com/No-Country-simulation/G9-LATAM-Team71/refs/heads/develop/data-science/csv_prueba_modelo.csv'
data = pd.read_csv(url)

data = data.drop(index=761)
data.info()

Separacion de variables, la variable a predecir (y) es la categoria que representa la descripción

In [ ]:
y = data.categoria
X = data.descripcion
y.value_counts()

Grupo de palabras que se usan para delimitar la oración, y quedan fuera de nuestra lista

In [ ]:
stop_words = [
    "de",
    "del",
    "la",
    "las",
    "el",
    "los",
    "en",
    "con",
    "para",
    "por",
    "y"
]

In [ ]:
X,X_test,y,y_test = train_test_split(X,y,stratify = y,random_state=5,test_size=0.15)#stratify indica que si hay un 10% de una variable, entonces ese 10 % tambien lo ponga en la division
X_train,X_val,y_train,y_val = train_test_split(X,y,stratify = y,random_state=5) #x de entrenamiento

vectorizer = TfidfVectorizer(stop_words=stop_words)

skf = StratifiedKFold(n_splits = 5,shuffle = True,random_state = 2)

In [ ]:
# funcion prueba modelo
# metricas
metrics = ['accuracy','recall_macro','f1_macro']


def probar_modelo(modelo, balanceo = None):
  pipeline = imbpipeline([('vectorizacion',vectorizer),('balanceo',balanceo),('modelo',modelo)])

  cv_resultados = cross_validate(pipeline,X,y,cv=skf,scoring=metrics)
  return cv_resultados

## Modelos a probar

* Logistic Regression
* Random Forest
* Linear SVC

### Logistic Regression

Es un modelo parecido a *Linear Regression* con la diferencia de que este modelo predice a partir de categorías y no de números como lo hace la regresión lineal

In [ ]:
regresion = LogisticRegression()
modelo_regresion_normal = probar_modelo(regresion);
modelo_regresion_balanceado = probar_modelo(regresion,SMOTE());
pd.DataFrame({
    "Regresion lineal sin balancear":pd.DataFrame(modelo_regresion_normal).mean(),
    "Regresion lineal balanceada":pd.DataFrame(modelo_regresion_balanceado).mean()
}).T

### Random Forest

In [ ]:
forest = RandomForestClassifier(max_depth=5)

forest_normal = probar_modelo(forest)
forest_smote = probar_modelo(forest,SMOTE())
pd.DataFrame({
    "RandomForest":pd.DataFrame(forest_normal).mean(),
    "RandomForest_balanceado":pd.DataFrame(forest_smote).mean()
}).T

### Linear SVC

In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
modelo_svm_normal = probar_modelo(svm)
modelo_svm_balanceado = probar_modelo(svm, SMOTE())
pd.DataFrame({
    "Linear SVC":pd.DataFrame(modelo_svm_normal).mean(),
    "Linear SVC balanceado":pd.DataFrame(modelo_svm_balanceado).mean()
}).T

## Comparación de modelos y modelo elegido

In [ ]:
pd.DataFrame({
    "Regresion lineal":pd.DataFrame(modelo_regresion_normal).mean(),
    "Regresion lineal balanceada":pd.DataFrame(modelo_regresion_balanceado).mean(),
    "RandomForest":pd.DataFrame(forest_normal).mean(),
    "RandomForest_balanceado":pd.DataFrame(forest_smote).mean(),
    "Linear SVC":pd.DataFrame(modelo_svm_normal).mean(),
    "Linear SVC balanceado":pd.DataFrame(modelo_svm_balanceado).mean()
}).T